# 05. Skill + MCP 통합 실습

이 노트북의 질문은 하나입니다. **MCP 도구 7개를 처음부터 모델에게 다 보여 주지 않고, 그 도구를 쓰는 방법을 적은 Skill을 읽은 뒤에만 보여 주면 무엇이 달라지는가?** 앞선 노트북에서 만든 요구사항 인터뷰 MCP 서버(도구 7개)와 Deep Agent를 그대로 쓰고, 이번에는 "도구가 언제 모델의 눈에 들어오는가"만 바꿉니다.

도구가 많으면 두 가지 비용이 생깁니다. 첫째, 모든 요청마다 도구 스키마가 입력 토큰으로 들어갑니다. 둘째, 지금 요청과 상관없는 도구가 눈앞에 있으면 모델이 쓸 필요 없는 도구를 부를 수 있습니다. Skill은 "언제, 어떤 순서로 이 도구들을 쓰는가"를 적어 둔 문서이므로, 그 문서를 읽은 모델에게만 도구를 보여 주는 것이 자연스러운 설계입니다. 이 설계가 실제로 유리한지는 말이 아니라 **작은 실측**으로 확인합니다.

이 노트북은 세 가지를 분리해서 봅니다.

1. **공개(disclosure)**: 모델이 도구 스키마를 보게 되는 시점. Skill을 읽으면 공개됩니다.
2. **권한(permission)**: 도구가 실제로 실행되도록 허락되는 조건. 공개와 권한은 다릅니다. 이 노트북은 그 차이를 코드로 보입니다.
3. **측정(measurement)**: 두 방식을 같은 사례에 같은 횟수로 돌려 비교하는 일. N이 작으므로 결과를 어떻게 읽어야 하는지까지 다룹니다.

## 학습 목표

- 이 과정의 `SKILL.md`에서 통합에 필요한 부분만 읽습니다: frontmatter, `metadata.include_tools`(공백으로 구분한 문자열), description 길이 한도, `SkillToolResolver`가 있다는 사실과 이 과정이 쓰지 않는다는 사실.
- 지연 공개(`defer_loading`)된 MCP 도구가 Skill을 읽기 전과 읽은 뒤에 모델 요청에서 어떻게 다른지 모델을 부르지 않고 확인합니다.
- 공개는 권한이 아니라는 것을 실행으로 보이고, 권한은 어디에서 강제해야 하는지 말합니다.
- 두 구성(`skill` 공개 / `upfront` 공개)을 같은 사례 2개에 3회씩 돌려 Wilson 구간과 작은 N의 한계까지 읽습니다.
- 절차를 MCP 서버에 둘지 Skill 폴더 안의 스크립트로 둘지 판단하는 기준을 말합니다.
- 06에서 다룰 Trigger Eval(Skill이 읽혔는가)과 Quality Eval(일이 제대로 됐는가)이 서로 다른 질문임을 압니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 0 | 사례와 질문: 도구 7개를 언제 보여 줄까 | 5 |
| 1 | 이 과정의 SKILL.md: 통합에 필요한 부분만 | 7 |
| 2 | 지연 공개: 읽기 전과 읽은 뒤 | 9 |
| 3 | 깨지는 SKILL.md: 한도와 형식 | 6 |
| 4 | 공개는 권한이 아니다 | 7 |
| 5 | 실측 비교: skill 공개 대 upfront 공개 | 10 |
| 6 | MCP 도구 대 Skill 스크립트, 그리고 06 미리보기 | 4 |
| 7 | 직접 해보기와 정리 | 2 |

## 호출 규모

5절의 비교 한 칸만 유료입니다. 위에서 아래로 한 번 실행하면 **Claude 호출 약 60~110회(실행 전 추정 상한 110회), 실행 12회**이고 2~5분이 걸립니다. 그 외의 모든 칸은 모델을 부르지 않습니다(서버를 띄우는 칸은 자식 프로세스 하나를 쓸 뿐입니다). Jev와 Langfuse는 이 노트북에서 호출하지 않습니다. `직접 해보기`의 유료 칸은 꺼져 있고(`RUN_EXERCISE = False`, `RUN_OPTIONAL = False`), 켜면 각각 약 110회, 약 180회 더 듭니다. 유료 칸은 실행 직전에 규모를 한 줄로 알립니다. 수강생 본인의 Anthropic 크레딧을 쓴다는 점을 기억하세요.

노트북의 커널 작업 폴더는 `Day04_new/`여야 `import cap_lab`이 됩니다. 이 노트북이 만드는 임시 파일은 `outputs/notebook/` 아래에만 생기며, 커밋 대상이 아닙니다.

In [ ]:
import dataclasses
import inspect
import json
import logging
from contextlib import contextmanager
from pathlib import Path

from deepagents.backends import FilesystemBackend
from deepagents.middleware.skills import MAX_SKILL_DESCRIPTION_LENGTH, SkillsMiddleware
from fastmcp.client.transports import PythonStdioTransport
from langchain.agents.middleware import ModelRequest
from langchain.agents.middleware.types import ModelResponse
from langchain.mcp import MCPAdapter
from langchain_anthropic import convert_to_anthropic_tool
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

from cap_lab.agent import SERVER_SCRIPT, SKILLS_DIR, server_argv
from cap_lab.experiment import Variant
from cap_lab.notebook import (
    announce, connect_server, key_table, new_workdir, pass_rate_table, run_small_compare, show_events, show_table,
)
from cap_lab.server.server import FaultPlan
from cap_lab.settings import ROOT, load_settings, make_model

# 키가 설정됐는지만 보여 줍니다. 값은 출력하지 않습니다.
settings = load_settings()
key_table(settings)

# 모델 객체를 만드는 것은 호출이 아닙니다. 아래 "모델이 보게 될 요청"을 만들 때 모델 종류가 필요해서 만들어 둡니다.
model = make_model(settings)

# 이 노트북이 만드는 임시 파일(고장 낸 Skill 사본, 서버 DB, 근거 폴더)은 모두 이 폴더 아래에 둡니다.
# Day04_new/skills/ 는 한 글자도 바꾸지 않습니다.
workdir = new_workdir("part5")

SKILL_FILE = SKILLS_DIR / "requirements-interview" / "SKILL.md"  # 디스크 위의 실제 경로
SKILL_PATH = "/requirements-interview/SKILL.md"  # 에이전트가 보는 가상 경로(cap_lab/agent.py:136에서 skills=["/"])
USER_REQUEST = "우리 팀 비품 대여 현황을 한눈에 보여 주는 도구를 만들고 싶어요. 어떤 걸 정해야 하는지 질문으로 정리해 주세요."


def fresh_folder(name: str) -> Path:
    """workdir 아래에 새 하위 폴더를 만든다. 서버마다 DB와 근거 폴더를 따로 쓰게 하려는 것이다."""
    folder = workdir / name
    folder.mkdir(parents=True)
    return folder


async def fetch_mcp_tools(folder: Path):
    """서버를 띄워 MCP 도구 7개를 가져온다. cap_lab/agent.py:110-114의 build_agent와 같은 연결 방식이다.

    모델은 부르지 않는다. 도구 목록만 읽고 곧바로 연결을 닫는다.
    """
    adapter = MCPAdapter(
        PythonStdioTransport(SERVER_SCRIPT, args=server_argv(folder / "sessions.sqlite3", folder / "evidence"),
                             cwd=str(ROOT))
    )
    try:
        tools = await adapter.list_tools()
    finally:
        await adapter.client.close()
    return tools


def mark_deferred(tool):
    """defer_loading을 켠 사본을 돌려준다. cap_lab/agent.py:148-152의 _deferred와 같은 한 줄이다."""
    return tool.model_copy(update={"extras": {**(tool.extras or {}), "defer_loading": True}})


def load_skills(skills_root: Path):
    """skills_root 아래 Skill들을 SkillsMiddleware가 읽는 대로 읽는다. (미들웨어, 상태)를 돌려준다.

    에이전트를 만들 때 create_deep_agent(skills=["/"])가 하는 일과 같다(cap_lab/agent.py:136).
    """
    backend = FilesystemBackend(root_dir=skills_root, virtual_mode=True)
    middleware = SkillsMiddleware(backend=backend, sources=["/"])
    state = middleware.before_agent({}, None, {})
    return middleware, state


@contextmanager
def capture_skill_warnings():
    """deepagents가 Skill을 읽다 남기는 경고(logging)를 모아서 보여 준다.

    경고는 모델에게 가지 않고 로그로만 남는다. 그래서 놓치기 쉽다.
    """
    logger = logging.getLogger("deepagents.middleware.skills")
    messages = []

    class Collector(logging.Handler):
        def emit(self, record):
            messages.append(record.getMessage())

    handler, old_propagate = Collector(level=logging.WARNING), logger.propagate
    logger.addHandler(handler)
    logger.propagate = False
    try:
        yield messages
    finally:
        logger.removeHandler(handler)
        logger.propagate = old_propagate


def conversation_after_reading(file_path: str, skill_text: str, status: str = "success", **read_args):
    """모델이 read_file(file_path, ...)을 부르고 결과를 받은 대화를 손으로 만든다.

    실제 에이전트가 남기는 메시지와 같은 모양(사용자 메시지, read_file 호출, 도구 결과)이다.
    read_args는 offset, limit처럼 read_file에 같이 넘기는 인자다.
    """
    call = AIMessage(content="", tool_calls=[
        {"name": "read_file", "args": {"file_path": file_path, **read_args}, "id": "call-read-1"}
    ])
    result = ToolMessage(content=skill_text, tool_call_id="call-read-1", status=status)
    return [HumanMessage(USER_REQUEST), call, result]


async def request_seen_by_model(skills_middleware, skills_state, messages, tools):
    """SkillsMiddleware를 거친 뒤 모델에게 실제로 가게 될 요청을 돌려준다. 모델은 부르지 않는다.

    미들웨어가 요청을 고친 다음 handler를 부르는데, 여기서는 handler가 그 요청을 붙잡기만 한다.
    """
    captured = {}

    async def capture(request):
        captured["request"] = request
        return ModelResponse(result=[AIMessage(content="")])

    request = ModelRequest(model=model, messages=messages, tools=list(tools),
                           state={**skills_state, "messages": messages}, runtime=None)
    await skills_middleware.awrap_model_call(request, capture)
    return captured["request"]


def tool_additions(request) -> list[dict]:
    """요청 메시지 안에 끼워 넣어진 tool_addition 블록의 도구 정의들."""
    return [
        block["tool"]["definition"]
        for message in request.messages if isinstance(message, SystemMessage)
        for block in message.content if isinstance(block, dict) and block.get("type") == "tool_addition"
    ]


def report(label: str, request) -> None:
    """모델 요청이 도구를 어떻게 담고 있는지 한 줄로 요약한다."""
    deferred = [tool.name for tool in request.tools if (tool.extras or {}).get("defer_loading") is True]
    added = [definition["name"] for definition in tool_additions(request)]
    kinds = [type(message).__name__ for message in request.messages]
    print(f"[{label}] 요청의 tools {len(request.tools)}개 중 defer_loading=True {len(deferred)}개 | "
          f"메시지 {kinds} | 끼워 넣어진 도구 정의 {len(added)}개 {added}")


print("준비 완료. 작업 폴더:", workdir.relative_to(ROOT), "| 모델 호출 0회")

## 0절. 사례와 질문: 도구 7개를 언제 보여 줄까 (5분)

과정 내내 같은 사례를 씁니다. 요구사항 인터뷰 MCP 서버입니다. 사용자가 "이런 도구를 만들고 싶어요"라고 말하면 에이전트가 질문으로 목표·제약·완료 기준을 정리하고, 서버가 그 답을 세션에 기록하고, 확정되면 명세(Seed)를 근거(Evidence) 파일로 남깁니다. 서버는 도구를 7개 가집니다.

그런데 사용자가 항상 이 일을 시키는 것은 아닙니다. "MCP가 뭐예요?"처럼 개념만 묻는 질문에는 인터뷰도 도구도 필요 없습니다. 이 둘이 이 노트북의 두 시험 사례입니다.

| 사례 | 사용자 요청 | 올바른 행동 |
|---|---|---|
| dev-01 | 비품 대여 현황 도구를 만들고 싶다, 질문으로 정리해 달라 | Skill을 읽고 `start_interview`를 부른다 (`should_trigger`) |
| dev-02 | MCP와 Seed가 뭔지 궁금하다, 뭘 만들려는 건 아니다 | Skill도 도구도 쓰지 않고 바로 답한다 (`should_not_trigger`) |

사례의 원문은 `data/cases.jsonl`의 두 줄이고, 기대값은 각 줄의 `expect`에 있습니다. 먼저 서버의 도구 7개를 실제로 불러 모델이 볼 수 있는 이름과 설명이 무엇인지 확인합니다. 서버는 `cap_lab/server/server.py`가 stdio 자식 프로세스로 뜨고, `cap_lab/notebook.py:76`의 `connect_server`가 `mcp.Client`로 연결합니다.

**먼저 예측하세요.** 도구 7개 가운데 dev-02(개념 질문)에 필요한 도구는 몇 개일까요? 그리고 dev-02에서도 7개의 스키마가 요청에 들어 있다면 모델이 그 가운데 하나를 불러 버릴 가능성이 있을까요? 노트에 적어 두고 5절에서 대조합니다.

In [ ]:
# [따라하기] 서버를 띄워 도구 7개의 이름과 설명을 본다 (모델 호출 0회)
folder = fresh_folder("tools-list")
async with connect_server(folder) as client:  # stdio 자식 프로세스로 cap_lab/server/server.py를 띄운다
    listing = await client.list_tools()  # MCP tools/list

tool_rows = [
    {"도구": tool.name, "인자": ", ".join(tool.input_schema.get("properties", {})), "설명": tool.description}
    for tool in listing.tools
]
show_table(tool_rows)
print("도구 수:", len(listing.tools), "| 모델 호출 0회")

## 1절. 이 과정의 SKILL.md: 통합에 필요한 부분만 (7분)

`SKILL.md`의 일반 형식(폴더 구조, 본문 쓰는 법)은 이 노트북의 주제가 아닙니다. 여기서는 **MCP 도구 공개와 맞닿는 세 줄**만 읽습니다. 이 과정의 Skill은 `skills/requirements-interview/SKILL.md`이고, 맨 위 frontmatter에 그 세 줄이 있습니다.

| frontmatter | 이 통합에서 하는 일 | 어기면 |
|---|---|---|
| `name` | Skill의 이름. 폴더 이름과 같아야 합니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:402`의 `_validate_skill_name`) | 경고를 남기고 계속 읽습니다 |
| `description` | **모델이 이 Skill을 읽을지 말지 정하는 유일한 근거**입니다. 시스템 프롬프트의 Skill 목록에는 이름, description, 읽을 경로만 올라가고 본문은 올라가지 않습니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:1099-1116`) | 1024자를 넘으면 앞 1024자만 남습니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:173`의 한도, 잘라내는 곳은 `.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:520-533`) |
| `metadata.include_tools` | 이 Skill을 읽으면 공개할 도구 이름들. **공백으로 구분한 문자열**입니다 | YAML 목록이나 쉼표 문자열이면 경고만 남기고 이름이 맞지 않아 아무것도 공개되지 않습니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:562-596`) |

두 가지를 짚습니다.

첫째, description은 **쓰는 조건과 쓰지 않는 조건을 모두** 담아야 합니다. 이 과정의 description은 "요구사항·명세를 정리하거나 근거 파일로 이어갈 때 사용한다"와 "MCP나 Seed의 개념만 묻는 질문에는 사용하지 않는다"를 함께 적었습니다. 뒤 문장이 dev-02를 위한 것입니다. 모델은 본문을 읽기 전에 description만 보고 읽을지 정하므로, 쓰지 않는 조건은 description에 있어야 합니다.

둘째, `include_tools`의 이름은 **에이전트에 넘긴 도구의 이름과 정확히 같아야** 합니다. 이름이 에이전트의 도구와 맞으면 그 도구가 지연 공개 대상이 됩니다(다음 절). 이름이 맞지 않는 도구가 필요할 때, 예를 들어 실행 시점에야 이름이 정해지는 MCP 도구는 `SkillToolResolver`(`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:176`)라는 함수로 이름을 도구로 바꿔 줄 수 있습니다. **이 과정은 resolver를 쓰지 않습니다.** 도구 7개의 이름이 Skill에 그대로 적혀 있고 `create_deep_agent`의 `tools=`로 이미 넘겼기 때문입니다 (`cap_lab/agent.py:121`).

아래 두 칸에서 실제 파일의 frontmatter를 보고, deepagents가 그것을 읽은 결과를 확인합니다.

In [ ]:
# [따라하기] 실제 SKILL.md의 frontmatter를 그대로 본다
text = SKILL_FILE.read_text(encoding="utf-8")
# 파일은 '---' 줄로 시작하고, 둘째 '---' 줄에서 frontmatter가 끝난다.
_, frontmatter, body = text.split("---\n", 2)
print(frontmatter)
print(f"frontmatter {len(frontmatter)}자 | 본문 {len(body)}자 | 전체 {len(text)}자")

In [ ]:
# [따라하기] deepagents가 이 Skill을 읽은 결과(skills_metadata)를 본다
real_middleware, real_state = load_skills(SKILLS_DIR)
skill = real_state["skills_metadata"][0]  # 이 과정에는 Skill이 하나뿐이다
include_tools = skill["metadata"]["include_tools"].split()  # 공백으로 나누면 도구 이름 목록이 된다
mcp_names = [tool.name for tool in listing.tools]  # 앞 칸에서 서버에서 받은 이름

show_table([
    {"항목": "name", "값": skill["name"]},
    {"항목": "description 글자 수", "값": f"{len(skill['description'])} / 한도 {MAX_SKILL_DESCRIPTION_LENGTH}"},
    {"항목": "include_tools 이름 수", "값": len(include_tools)},
    {"항목": "include_tools 가 서버 도구와 같은 집합인가", "값": set(include_tools) == set(mcp_names)},
    {"항목": "경고", "값": real_state["skills_load_errors"] or "없음"},
])
print("description 전문:", skill["description"])

## 2절. 지연 공개: 읽기 전과 읽은 뒤 (9분)

`cap_lab/agent.py:115-116`은 `tool_disclosure="skill"`일 때 MCP 도구마다 사본을 만들어 `extras={"defer_loading": True}`를 켭니다 (`cap_lab/agent.py:148-152`의 `_deferred`). 이 표지가 붙은 도구는 모델 요청에 **스키마 없이** 실립니다. deepagents의 `SkillsMiddleware`가 대화에서 `read_file`로 SKILL.md를 읽은 기록을 찾으면, 그 Skill의 `include_tools`에 적힌 이름의 도구를 공개합니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:1322-1365`, 공개를 계획하는 곳은 `.venv/lib/python3.13/site-packages/deepagents/middleware/_skill_tools.py:194`의 `_plan_disclosure`).

공개가 요청에 어떻게 나타나는지는 모델 종류에 따라 다릅니다. 이 과정의 `ChatAnthropic`은 **요청의 `tools` 목록을 바꾸지 않고**, 대화 안에 `tool_addition` 블록을 담은 `SystemMessage`를 끼워 넣습니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/_skill_tools.py:320`의 `_insert_disclosures`). 도구 목록을 직접 늘려 바인딩하는 길(`.venv/lib/python3.13/site-packages/deepagents/middleware/_skill_tools.py:342`의 `_bind_disclosures`)은 이런 방식을 지원하지 않는 모델용입니다. 그래서 "읽은 뒤에 도구가 늘어난다"는 말은 이 과정에서는 정확하지 않고, "읽은 뒤에 도구 정의가 대화에 추가된다"가 정확합니다.

또 하나. Anthropic에게 보내는 도구 정의에 `defer_loading`이 어떻게 실리는지는 `langchain_anthropic`의 `convert_to_anthropic_tool`이 정합니다 (`.venv/lib/python3.13/site-packages/langchain_anthropic/chat_models.py:3227-3236`, `extras`의 `defer_loading`을 최상위 필드로 옮깁니다). 아래 첫 칸에서 직접 확인합니다.

In [ ]:
# [따라하기] 지연 공개 표지가 Anthropic 도구 정의에 어떻게 실리는지 본다 (모델 호출 0회)
mcp_tools = await fetch_mcp_tools(fresh_folder("agent-tools"))  # 서버에서 도구 7개를 가져온다
deferred_tools = [mark_deferred(tool) for tool in mcp_tools]  # 모두 defer_loading=True 사본으로 바꾼다

plain_form = convert_to_anthropic_tool(mcp_tools[0])  # 표지 없는 원본
deferred_form = convert_to_anthropic_tool(deferred_tools[0])  # 표지 있는 사본
print("원본 도구 정의의 키:", sorted(plain_form))
print("사본 도구 정의의 키:", sorted(deferred_form))
print("사본의 defer_loading:", deferred_form.get("defer_loading"))
print("원본 도구에 표지가 없는지:", (mcp_tools[0].extras or {}).get("defer_loading") is None)

**먼저 예측하세요.** 같은 사용자 요청으로 모델에게 보낼 요청을 두 번 만듭니다. 한 번은 대화에 사용자 메시지만 있을 때, 한 번은 모델이 `read_file`로 SKILL.md를 읽고 결과를 받은 뒤입니다. (1) 두 요청의 `tools` 목록 길이는 같을까요, 다를까요? (2) `defer_loading=True`인 도구의 수는? (3) 대화에 끼워 넣어진 도구 정의는 몇 개이고 어느 위치(어느 메시지 뒤)일까요? 적어 둔 뒤 실행하세요.

아래 칸의 "읽은 뒤" 대화는 에이전트가 만드는 메시지와 같은 모양으로 **손으로 만든 것**입니다. 실제 모델이 읽었는지가 아니라, "읽은 기록이 있으면 미들웨어가 무엇을 하는가"만 보입니다. 실제 모델이 읽는지는 5절의 실측이 봅니다.

In [ ]:
# [따라하기] 읽기 전과 읽은 뒤, 모델에게 갈 요청을 나란히 만든다 (모델 호출 0회)
skill_text = SKILL_FILE.read_text(encoding="utf-8")

before_messages = [HumanMessage(USER_REQUEST)]  # 아직 아무것도 읽지 않았다
after_messages = conversation_after_reading(SKILL_PATH, skill_text)  # read_file(SKILL.md) 뒤

request_before = await request_seen_by_model(real_middleware, real_state, before_messages, deferred_tools)
request_after = await request_seen_by_model(real_middleware, real_state, after_messages, deferred_tools)

report("읽기 전", request_before)
report("읽은 뒤", request_after)

# 끼워 넣어진 도구 정의 하나를 자세히 본다. record_answer의 field 인자에 enum이 그대로 있는지 확인한다.
record_answer = next(d for d in tool_additions(request_after) if d["name"] == "record_answer")
print("record_answer.field 스키마:", json.dumps(record_answer["input_schema"]["properties"]["field"], ensure_ascii=False))
print("끼워 넣어진 정의에 defer_loading이 남아 있는지:", "defer_loading" in record_answer)

**관찰할 것.** 세 가지를 표시하세요.

1. 두 요청 모두 `tools`는 7개이고 모두 `defer_loading=True`입니다. 읽은 뒤에도 `tools` 목록은 그대로입니다. 바뀐 것은 대화에 `SystemMessage`가 하나 생긴 것이고, 그 안에 도구 정의 7개가 들어 있습니다. 위치는 `ToolMessage`(읽기 결과) 바로 뒤입니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/_skill_tools.py:303`의 `_insertion_point`: 읽기 결과 묶음 뒤에 끼웁니다).
2. 끼워 넣어진 정의는 `field`의 `enum`을 보존합니다. 즉 공개되는 것은 스키마 전체이며, 02에서 다룬 인터페이스 품질이 그대로 모델에게 갑니다.
3. 끼워 넣어진 정의에는 `defer_loading`이 없습니다. 공개된 도구는 더 이상 지연 대상이 아닙니다.

이 칸은 모델을 한 번도 부르지 않았습니다. 그러니 "Skill을 읽으면 도구가 공개된다"는 **메커니즘**은 확인했지만, "모델이 실제로 Skill을 읽는가", "읽으면 더 잘 쓰는가"는 아직 아무것도 모릅니다. 그 구분이 이 노트북 전체의 골격입니다.

In [ ]:
# [바꿔보기] 무엇이 "읽은 것"으로 인정되는지 바꿔 본다. TRIALS를 고쳐서 다시 실행하세요 (모델 호출 0회)
TRIALS = {
    "SKILL.md 전체 읽기": dict(file_path=SKILL_PATH),
    "SKILL.md 앞 3줄만 읽기(offset/limit)": dict(file_path=SKILL_PATH, offset=0, limit=3),
    "다른 파일 읽기": dict(file_path="/requirements-interview/notes.md"),
    "SKILL.md 읽기가 오류로 끝남": dict(file_path=SKILL_PATH, status="error"),
}

trial_rows = []
for label, arguments in TRIALS.items():
    messages = conversation_after_reading(skill_text=skill_text, **arguments)
    request = await request_seen_by_model(real_middleware, real_state, messages, deferred_tools)
    trial_rows.append({"시도": label, "공개된 도구 수": len(tool_additions(request))})
show_table(trial_rows)

**관찰할 것.** 앞 3줄만 읽어도 7개가 공개되고, 오류로 끝난 읽기와 다른 파일 읽기는 0개입니다. 규칙은 `.venv/lib/python3.13/site-packages/deepagents/middleware/_skill_tools.py:265`의 `_find_skill_reads` 문서에 적혀 있습니다. 성공한 `read_file` 결과의 경로가 Skill 파일과 같으면 읽은 것으로 칩니다. `offset`이나 `limit`이 있어도 읽은 것이고, 내용이 나중에 잘렸어도 읽은 것입니다. **본문 전체를 정말 읽었는지는 보지 않습니다.** "읽었다"의 정의가 이렇게 느슨하다는 점은 06에서 Trigger Eval을 정의할 때 다시 나옵니다.

## 3절. 깨지는 SKILL.md: 한도와 형식 (6분)

SKILL.md가 잘못되었을 때 이 통합은 **조용히** 깨집니다. 예외가 나지 않고 경고가 로그로만 남으며, 에이전트는 평소처럼 돌아가지만 도구가 공개되지 않거나 description의 일부가 사라집니다. 두 가지를 일부러 깨 봅니다.

고장 낸 사본은 `skills/`가 아니라 이 노트북의 작업 폴더에 씁니다. 실제 파일은 건드리지 않습니다. 사본의 폴더 이름은 `name:`과 같아야 합니다(`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:402`).

**먼저 예측하세요.** description이 1024자를 넘으면 deepagents는 (a) 예외를 내고 Skill을 버린다, (b) 앞 1024자만 남기고 계속한다, (c) 전체를 그대로 쓴다 중 무엇을 할까요? 그리고 description 앞쪽에 군더더기가 길게 있고 "쓰지 않는 조건"이 맨 뒤에 있다면 잘려 나가는 것은 무엇일까요?

In [ ]:
# 고장 낸 사본을 만드는 도우미. 실제 SKILL.md의 본문은 그대로 쓰고 frontmatter만 바꾼다.
def write_skill_copy(folder_name: str, *, description: str, include_tools_lines: str) -> Path:
    """workdir 아래에 requirements-interview/SKILL.md 사본을 쓰고 그 위쪽 폴더(skills 루트)를 돌려준다.

    description은 JSON 문자열로 써서 YAML 큰따옴표 문자열이 되게 한다. include_tools_lines는
    metadata 아래에 들어갈 줄들이므로 들여쓰기까지 직접 넘긴다.
    """
    root = workdir / folder_name
    skill_dir = root / "requirements-interview"  # 폴더 이름이 name과 같아야 한다
    skill_dir.mkdir(parents=True)
    head = (
        "---\n"
        "name: requirements-interview\n"
        f"description: {json.dumps(description, ensure_ascii=False)}\n"
        "metadata:\n"
        f"{include_tools_lines}\n"
        "---\n"
    )
    (skill_dir / "SKILL.md").write_text(head + body, encoding="utf-8")  # body는 1절 칸에서 읽어 둔 실제 본문
    return root


print("도우미 준비 완료. 쓸 위치:", workdir.relative_to(ROOT))

In [ ]:
# [깨뜨리기] description 앞에 군더더기를 길게 붙여 1024자 한도를 넘긴다 (모델 호출 0회)
original_description = skill["description"]  # 1절에서 읽은 실제 description (쓰지 않는 조건이 끝에 있다)
padding = "이 문장은 한도를 시험하려고 덧붙인 군더더기입니다. " * 45  # 앞쪽을 채운다
broken_description = padding + original_description
include_tools_line = f'  include_tools: "{skill["metadata"]["include_tools"]}"'  # 도구 이름은 그대로 둔다

broken_root = write_skill_copy("broken-description", description=broken_description,
                               include_tools_lines=include_tools_line)
with capture_skill_warnings() as warnings:
    _, broken_state = load_skills(broken_root)
loaded = broken_state["skills_metadata"][0]

show_table([
    {"항목": "붙인 뒤 description 길이", "값": len(broken_description)},
    {"항목": "읽힌 description 길이", "값": len(loaded["description"])},
    {"항목": "'사용하지 않는다' 가 남았나", "값": "사용하지 않는다" in loaded["description"]},
    {"항목": "예외가 났나", "값": False},
])
print("skills_load_errors:", broken_state["skills_load_errors"])
print("로그로만 남은 경고:", warnings)

**관찰할 것.** 정답은 (b)입니다. 길이는 정확히 1024자로 줄었고 예외는 없었으며, 경고는 로그로만 나왔습니다. 잘려 나간 것은 **끝**이므로 "MCP나 Seed의 개념만 묻는 질문에는 사용하지 않는다"라는 쓰지 않는 조건이 통째로 사라졌습니다. 이 Skill로는 dev-02에서 모델이 Skill을 읽을 이유를 막을 근거가 없어집니다. 모델이 보는 description은 잘린 결과이므로, 쓰는 조건은 앞에, 쓰지 않는 조건도 앞쪽에 두는 것이 안전합니다. 한도 근처에서 쓰는 사람이 손해를 보게 되어 있습니다.

경고가 `skills_load_errors`가 아니라 로그로만 갔다는 점도 표시하세요. 위 칸이 출력한 `skills_load_errors`는 비어 있습니다. 한도 초과를 잡는 일은 deepagents가 아니라 **우리가 하는 검사**여야 합니다. 06에서 이런 검사를 결정적 scorer로 만드는 이야기를 합니다.

In [ ]:
# [깨뜨리기] include_tools를 YAML 목록으로 쓴다. 문자열이어야 하는 자리에 목록을 넣는다 (모델 호출 0회)
list_lines = "  include_tools:\n" + "\n".join(f"    - {name}" for name in include_tools)  # 목록 형식
list_root = write_skill_copy("broken-include-tools", description=original_description,
                             include_tools_lines=list_lines)
with capture_skill_warnings() as warnings:
    list_middleware, list_state = load_skills(list_root)

coerced = list_state["skills_metadata"][0]["metadata"]["include_tools"]
print("읽힌 include_tools:", repr(coerced)[:80], "...")
print("공백으로 나눈 첫 이름:", coerced.split()[0])
print("로그로만 남은 경고:", [warning[:90] for warning in warnings])

# 이제 모델이 SKILL.md를 정상적으로 읽은 대화를 만들어, 그래도 도구가 공개되는지 본다.
list_text = (list_root / "requirements-interview" / "SKILL.md").read_text(encoding="utf-8")
messages = conversation_after_reading(SKILL_PATH, list_text)
request = await request_seen_by_model(list_middleware, list_state, messages, deferred_tools)
report("목록 형식 include_tools, 읽은 뒤", request)

**관찰할 것.** YAML 목록은 deepagents가 `str()`로 바꿔 `"['start_interview', 'record_answer', ...]"`라는 한 덩어리 문자열로 읽습니다. 공백으로 나누면 첫 이름이 `['start_interview',`처럼 괄호와 따옴표가 붙어 있어 어떤 도구 이름과도 맞지 않습니다. 그래서 모델이 SKILL.md를 정확히 읽었는데도 **공개된 도구는 0개**입니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:562-596`, 경고 문구는 "should be a space-separated string"입니다).

이 고장의 무서운 점은 증상이 "도구가 안 보이는 것"이 아니라 "에이전트가 도구가 없다고 판단해 말로만 대답하는 것"으로 나타난다는 점입니다. 점수로는 `tool_selection_ok` 실패로만 잡히고, 원인이 SKILL.md 형식이라는 것은 로그를 열어야 압니다. 원인을 가르려면 `trigger_ok`(읽었는가)와 `tool_selection_ok`(불렀는가)를 따로 봐야 합니다. 6절 끝에서 이어집니다.

## 4절. 공개는 권한이 아니다 (7분)

지금까지 "Skill을 읽어야 도구가 보인다"를 봤습니다. 흔한 오해는 여기서 "그러면 읽기 전에는 도구를 못 쓴다"로 넘어가는 것입니다. **숨기는 것과 막는 것은 다릅니다.**

세 가지 사실을 코드에서 확인했습니다.

- `defer_loading`은 모델에게 보내는 **스키마**를 감출 뿐입니다. 도구 자체는 에이전트의 도구 목록에 등록되어 있습니다 (`cap_lab/agent.py:121`의 `tools=tools`).
- deepagents가 "읽기 전에는 호출이 실패한다"고 보장하는 것은 `SkillsMiddleware`에 `tools=`로 넘긴 **skill tool**입니다. 이 도구는 도구 노드에 등록되지 않고, 공개된 동안에만 호출이 통과합니다 (`skills.py` 클래스 문서의 "Skill tools" 절, `.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:1375`의 `wrap_tool_call`). 반면 에이전트에 직접 넘긴 지연 도구는 이미 등록되어 있어 `request.tool`이 비어 있지 않으므로 그대로 통과합니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:1389-1393`). `disclosed_skill_tool_names`도 "에이전트 자신의 지연 도구는 들어가지 않는다"고 적습니다 (`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:1416-1429`).
- cap_lab은 후자(에이전트에 직접 넘긴 지연 도구)를 씁니다. 그래서 모델이 SKILL.md를 읽기 전에 도구 이름만 알고 부르면, 그 호출은 서버까지 갑니다. 이 동작은 `tests/test_variants.py`의 `test_disclosure_changes_only_what_the_model_sees_and_a_called_tool_runs_under_both`가 시험합니다.

**먼저 예측하세요.** Skill을 읽지 않은 채 MCP 서버에 `start_interview`를 직접 부르면 서버는 어떻게 할까요? (a) "Skill을 먼저 읽으세요" 오류 (b) 정상 실행. 서버는 Skill이라는 것을 알까요?

In [ ]:
# [깨뜨리기] Skill도 에이전트도 없이 MCP 서버를 직접 부른다. 서버는 Skill을 모른다 (모델 호출 0회)
folder = fresh_folder("no-skill-call")
async with connect_server(folder) as client:
    started = await client.call_tool("start_interview", {"goal": "비품 대여 현황을 보여 주는 도구 만들기"})

result = started.structured_content
print("호출이 실패로 표시됐나(isError):", started.is_error)
print("업무 결과 success:", result["success"], "| status:", result["status"])
print("서버가 만든 세션 앞 8자리:", result["session_id"][:8])
show_events(folder / "evidence")  # 서버가 근거 폴더에 남긴 이벤트

**관찰할 것.** 정답은 (b)입니다. 서버는 정상 실행했고 근거 이벤트까지 남겼습니다. 서버에는 "Skill을 읽었는가"를 판단할 정보가 없습니다. 한계도 분명히 하겠습니다. 이 칸은 에이전트를 거치지 않고 클라이언트로 직접 부른 것이므로, "에이전트가 SKILL.md를 읽기 전에 부른 호출이 실제로 서버까지 간다"를 이 칸이 증명하지는 않습니다. 그 부분은 위에서 말한 시험(`tests/test_variants.py`)과 deepagents 코드(`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:1389-1393`)를 근거로 합니다. 이 노트북에서 직접 실행해 확인한 것은 "서버는 Skill을 모른다"까지입니다.

그렇다면 권한은 어디에서 강제하나요? 호출이 **실제로 실행되는 곳**에서 막아야 합니다.

| 강제하는 곳 | 방법 | 이 저장소에서 확인한 것 |
|---|---|---|
| MCP 서버 | 서버 코드가 호출을 거절합니다. 클라이언트가 누구든, Skill을 읽었든 상관없습니다 | `cap_lab/server/server.py:64-103`의 `FaultPlan.permission_denied`, 아래 칸에서 실행 |
| 에이전트 미들웨어 | `wrap_tool_call`에서 호출을 가로채 통과시키거나 막습니다 | `cap_lab/middleware.py:52`의 `GoalGuard`가 `start_interview`만 검사합니다(07에서 다룹니다) |
| 사람 승인 | 호출 전에 `interrupt`로 사람의 허락을 받습니다 | `langchain.agents.middleware.HumanInTheLoopMiddleware(interrupt_on=...)`가 설치되어 있고 시그니처를 확인했습니다. 이 과정은 쓰지 않습니다 |
| deepagents skill tool | `SkillsMiddleware(tools=...)`로 넘긴 도구는 읽기 전에 호출이 실패한다고 클래스 문서가 적습니다 | **확인 필요.** 문서와 코드는 읽었지만 이 과정은 이 경로를 쓰지 않아 실행해 보지 않았습니다 |
| 파일 접근 | `FilesystemPermission`으로 파일 읽기·쓰기를 거절합니다. 도구 호출이 아니라 **파일 작업**에 대한 것입니다 | `cap_lab/agent.py:137`가 쓰기를 모두 거절합니다 |

이 표에 없는 방법을 지어내지 마세요. 특히 "Skill에 '읽기 전에는 쓰지 말라'고 적으면 막힌다"는 것은 권한이 아니라 **부탁**입니다. 아래 칸은 서버 쪽 강제가 Skill과 무관하게 작동하는 모습을 보입니다.

In [ ]:
# [바꿔보기] 서버가 도구 하나를 권한 없음으로 거절하게 하고, 같은 서버의 다른 도구와 비교한다 (모델 호출 0회)
# plan 을 바꿔 보세요. 예: FaultPlan(permission_denied=["freeze_seed"])
plan = FaultPlan(permission_denied=["save_evidence"])
folder = fresh_folder("denied-call")
async with connect_server(folder, fault=plan) as client:
    started = await client.call_tool("start_interview", {"goal": "비품 대여 현황을 보여 주는 도구 만들기"})
    session_id = started.structured_content["session_id"]
    denied = await client.call_tool("save_evidence", {"session_id": session_id, "title": "시험", "content": "본문"})

print("start_interview:", started.structured_content["success"])
print("save_evidence isError:", denied.is_error, "| 결과:", denied.structured_content)

**관찰할 것.** `start_interview`는 성공했고 `save_evidence`는 `permission_denied`로 거절되었습니다. 두 호출 모두 Skill을 읽은 적이 없습니다. 또 `isError`는 `False`입니다. 거절은 프로토콜 오류가 아니라 **업무 오류**로 돌아옵니다 (`success=false`와 `error` 코드). 그래서 에이전트가 이 결과를 읽고 "같은 호출을 반복하지 않는다"고 판단해야 하며, SKILL.md의 `permission_denied` 항목이 바로 그 지침입니다. 06에서 이 반응을 `no_blind_retry`로 잽니다.

정리하면 Skill은 **모델에게 알려 주는 층**이고, 서버와 미들웨어는 **강제하는 층**입니다. 지연 공개는 첫 번째 층에서 토큰과 선택지를 줄이는 장치이지, 두 번째 층을 대신하지 않습니다.

## 5절. 실측 비교: skill 공개 대 upfront 공개 (10분)

이제 "Skill을 읽은 뒤에만 공개하면 무엇이 달라지는가"를 재 봅니다. 비교하는 두 구성은 `Variant`(`cap_lab/experiment.py:66`)의 `tool_disclosure` 하나만 다릅니다.

| 구성 | `tool_disclosure` | 첫 Claude 요청의 MCP 도구 7개 |
|---|---|---|
| `skill` | `"skill"` (기본값) | 모두 `defer_loading: true`. Skill을 읽은 뒤에 공개 (`cap_lab/agent.py:115-116`) |
| `upfront` | `"upfront"` | 표지 없이 처음부터 모두 보냄 |

다른 것은 모두 같습니다. 같은 서버, 같은 SKILL.md(두 구성 모두 Skill 목록은 그대로 시스템 프롬프트에 있고 Skill 폴더도 그대로입니다), 같은 모델, 게이트는 모두 꺼짐. 그래서 차이가 있다면 공개 방식 때문이라고 말할 수 있는 구조입니다.

사례는 dev-01(`should_trigger`)과 dev-02(`should_not_trigger`), 각각 3회, 두 구성이므로 2 x 3 x 2 = **12회 실행**입니다. 한 번 실행에 Claude가 대략 5~9번 불리므로 합계 약 60~110회입니다 (`docs/DESIGN.md` 8절).

읽을 숫자는 셋입니다.

- `trigger_ok` (`cap_lab/scorers.py:123`): Skill 본문을 **읽어야 하는 사례는 읽었고, 읽지 말아야 하는 사례는 읽지 않았는가**.
- `tool_selection_ok` (`cap_lab/scorers.py:134`): 필수 도구는 모두 불렀고 금지 도구는 부르지 않았는가.
- 실행당 `input_tokens` 평균.

한 가지를 미리 밝힙니다. `upfront`에서는 모델이 Skill을 읽지 않고도 도구를 쓸 수 있으므로, dev-01의 `trigger_ok`(읽었는가)가 실패해도 일 자체는 성공할 수 있습니다. 그래서 `upfront`의 `trigger_ok`는 "잘못"이라기보다 **서술**입니다. 이 구성에서 읽는 행위가 설계의 일부가 아니기 때문입니다.

**먼저 예측하세요.** 다음 네 칸을 채워 두세요. (1) `skill`과 `upfront` 가운데 dev-02에서 `start_interview`를 잘못 부를 가능성이 큰 쪽은? (2) dev-01의 입력 토큰 평균은 어느 쪽이 더 클까요? (읽는 턴이 하나 더 생기는 쪽 / 스키마 7개를 매번 싣는 쪽) (3) dev-02의 입력 토큰 평균은? (4) 3회 반복으로 두 구성의 차이를 확정할 수 있을까요? 결과를 보기 전에 적는 것이 이 칸의 핵심입니다.

In [ ]:
# [따라하기] 두 구성을 만든다. tool_disclosure 외에는 같다는 것을 코드로 확인한다 (모델 호출 0회)
variant_skill = Variant(name="skill", tool_disclosure="skill")
variant_upfront = Variant(name="upfront", tool_disclosure="upfront")

as_skill, as_upfront = dataclasses.asdict(variant_skill), dataclasses.asdict(variant_upfront)
differences = {key: (as_skill[key], as_upfront[key]) for key in as_skill if as_skill[key] != as_upfront[key]}
print("두 구성이 다른 필드:", differences)
assert set(differences) == {"name", "tool_disclosure"}, "name(작업 폴더 이름) 외에 tool_disclosure만 달라야 합니다"

compare_rows = []  # 아래 유료 칸이 채웁니다. 실행하지 않았다면 비어 있습니다.

In [ ]:
# [따라하기] 12회 실행: dev-01, dev-02 x 3회 x 두 구성
announce("Part 5 비교: skill 공개 대 upfront 공개", model_calls=110, runs=12)
compare_rows = run_small_compare(
    [variant_skill, variant_upfront],  # 변형 이름이 작업 폴더 이름이 되므로 서로 달라야 한다
    ["dev-01", "dev-02"],  # dev 사례만 허용된다
    3,  # 반복 횟수
    label="part5-disclosure",
)
print("행 수:", len(compare_rows), "| 오류 행:", sum(row["error"] is not None for row in compare_rows))

In [ ]:
# [따라하기] 통과율과 Wilson 95% 구간, 그리고 실행당 입력 토큰 평균
def mean_by_variant(rows: list[dict], name: str, case_id: str | None = None) -> dict[str, float | None]:
    """변형마다 name 점수의 평균을 구한다. case_id를 주면 그 사례만 본다. 오류 행(값이 None)은 뺀다."""
    means = {}
    for variant in dict.fromkeys(row["variant"] for row in rows):
        values = [row[name] for row in rows
                  if row["variant"] == variant and (case_id is None or row["case_id"] == case_id)
                  and row.get(name) is not None]
        means[variant] = round(sum(values) / len(values)) if values else None
    return means


if compare_rows:
    rate_rows = pass_rate_table(compare_rows, ["trigger_ok", "tool_selection_ok"])
    show_table(rate_rows)  # passed/checked는 통과 수/확인한 수, unchecked는 확인 불가(None) 수
    token_rows = [{"사례": case_id or "전체", **mean_by_variant(compare_rows, "input_tokens", case_id)}
                  for case_id in (None, "dev-01", "dev-02")]
    show_table(token_rows)
else:
    print("compare_rows가 비어 있습니다. 앞의 유료 칸을 실행하지 않았습니다.")

**관찰할 것.** 표를 읽는 순서를 정해 두겠습니다.

1. **먼저 `unchecked`와 오류 행을 봅니다.** 오류가 난 실행은 값이 `None`이고 통과로도 실패로도 세지 않습니다. 12회 중 몇 회가 오류였는지가 비교의 분모를 바꿉니다.
2. **통과율은 `passed/checked`로 읽습니다.** 구성마다 점수당 확인한 수는 최대 6입니다(2 사례 x 3회). 6개 중 6개가 통과여도 Wilson 95% 구간의 하한은 약 0.61, 5개면 0.44~0.97, 3개면 0.19~0.81입니다. 이 구간들은 **서로 크게 겹칩니다.** 즉 두 구성의 통과율이 6/6 대 5/6으로 달라 보여도, 이 표만으로는 어느 쪽이 낫다고 말할 수 없습니다.
3. **입력 토큰은 평균이지만 N도 작습니다.** 사례별 값을 보세요. dev-01은 Skill을 읽는 턴이 있고 dev-02는 읽지 않는 것이 기대값이므로, 두 사례의 토큰 구조가 다릅니다. 전체 평균 하나는 이 둘을 섞어 어느 쪽 이야기도 흐립니다.
4. **예측과 대조합니다.** 적어 둔 네 가지 가운데 틀린 것은 무엇이었는지 표시하세요. 틀린 예측이 가장 값진 정보입니다.

이 비교는 **탐색**입니다. 사전 등록한 판정 규칙(`docs/DESIGN.md`의 P5, A/A 잡음 폭을 거치는 쌍체 비교)을 거친 것이 아니고, 같은 구성을 두 번 돌려 잡음 폭을 재지도 않았습니다. 그러므로 결과는 "이 12회에서 관찰한 것"이지 "skill 공개가 낫다/나쁘다"가 아닙니다. 3회 미만 반복은 '1회 관찰'로 부르는 것이 이 과정의 약속인데, 여기서는 3회이므로 관찰이 아니라 아주 작은 표본입니다. 이 구분을 07에서 A/A와 쌍체 비교로 정식화합니다.

In [ ]:
# [바꿔보기] 요약 대신 실행 하나하나를 본다. 작은 N에서는 구간보다 행을 읽는 편이 낫다 (모델 호출 0회)
# 보고 싶은 열을 바꿔 보세요. 예: "first_failure"에 값이 있는 행만 골라 보기
if compare_rows:
    columns = ["variant", "repeat", "case_id", "trigger_ok", "tool_selection_ok", "first_failure", "error",
               "input_tokens", "model_calls"]
    show_table(compare_rows, columns=columns)
else:
    print("compare_rows가 비어 있습니다.")

**관찰할 것.** 같은 구성·같은 사례의 3회가 서로 얼마나 다른지 보세요. `input_tokens`와 `model_calls`가 같은 조건에서도 흔들린다면 그 흔들림이 구성 간 차이를 가릴 수 있다는 뜻입니다. `first_failure`가 있는 행은 6단계 순서(S1 Skill 트리거, S2 시작 인자, …)에서 **가장 먼저** 실패한 단계입니다. dev-01과 dev-02는 모두 S1 사례이므로 실패가 있다면 대개 S1 또는 S2입니다.

행마다 `evidence_dir`가 있어서, 어떤 도구가 어떤 순서로 불렸는지 `show_events`로 다시 볼 수 있습니다. 이상한 행이 있으면 숫자를 해석하기 전에 그 행의 근거 폴더부터 여세요.

## 6절. MCP 도구 대 Skill 스크립트, 그리고 06 미리보기 (4분)

Skill 폴더에는 SKILL.md 말고 스크립트나 참고 파일도 넣을 수 있습니다. 그러면 절차를 MCP 서버에 둘지, Skill 폴더의 스크립트로 둘지 고민이 생깁니다. 이 과정이 쓰는 기준은 다음과 같습니다. 이것은 **이 수업의 판단 기준**이지 명세가 아닙니다.

| 질문 | MCP 서버에 둔다 | Skill 폴더의 스크립트로 충분하다 |
|---|---|---|
| 상태가 대화 밖에 남아야 하나 | 예. 세션과 근거 파일이 서버 DB에 남아 다음 실행이 이어받습니다 | 아니오. 한 번 돌고 끝납니다 |
| 여러 클라이언트·에이전트가 같은 계약을 써야 하나 | 예. 도구 스키마가 계약이고 `tools/list`로 공개됩니다 | 아니오. 이 Skill을 읽은 에이전트 하나만 씁니다 |
| 입력을 검증하고 거절해야 하나 | 예. 서버가 `permission_denied`나 `criteria_not_testable`처럼 거절합니다 | 아니오. 틀려도 되돌릴 수 있습니다 |
| 부작용이 있나(저장, 전송, 결제) | 예. 서버에서 강제하고 기록합니다 | 아니오. 계산이나 서식 같은 순수한 변환입니다 |
| 실행 수단이 에이전트에 있나 | 서버가 실행합니다. 에이전트는 호출만 합니다 | **스크립트를 돌릴 도구가 있어야 합니다.** 이 과정의 에이전트는 파일 도구 4개(`ls`, `read_file`, `glob`, `grep`)만 열려 있어(`cap_lab/agent.py:46`) 스크립트를 실행할 수 없습니다 |

마지막 줄이 현실적인 제약입니다. Skill의 스크립트는 "읽을 수 있는 파일"이지 "부를 수 있는 도구"가 아닙니다. 에이전트에 실행 도구(셸이나 코드 실행)를 주는 순간 권한 문제가 새로 생깁니다. 반면 MCP 도구는 이미 호출 계약과 거절 경로를 가집니다. 그래서 상태·검증·부작용이 있는 절차는 서버에, 문장 서식을 맞추는 정도의 순수 변환은 Skill의 참고 자료로 두는 것이 이 과정의 기본선입니다.

In [ ]:
# [바꿔보기] 절차 몇 개를 기준에 넣어 본다. SCENARIOS의 값을 바꾸거나 새 절차를 추가해 보세요 (모델 호출 0회)
def where_does_it_belong(needs_state: bool, shared_contract: bool, must_validate: bool, has_side_effect: bool) -> str:
    """이 수업의 기준(위 표)으로 절차의 자리를 정한다. 하나라도 예이면 MCP 서버다.

    명세가 아니라 수업용 휴리스틱이다. 답과 함께 어느 기준이 예였는지 이유를 돌려준다.
    """
    reasons = [name for name, flag in (("상태", needs_state), ("공유 계약", shared_contract),
                                       ("검증·거절", must_validate), ("부작용", has_side_effect)) if flag]
    return f"MCP 서버 ({', '.join(reasons)})" if reasons else "Skill 폴더의 스크립트나 참고 자료"


SCENARIOS = {
    "세션에 답변 기록 (record_answer)": dict(needs_state=True, shared_contract=True, must_validate=True, has_side_effect=True),
    "명세 문서의 제목 서식 맞추기": dict(needs_state=False, shared_contract=False, must_validate=False, has_side_effect=False),
    "근거 파일 저장 (save_evidence)": dict(needs_state=True, shared_contract=True, must_validate=True, has_side_effect=True),
    "질문 목록을 표로 정렬해 보여 주기": dict(needs_state=False, shared_contract=False, must_validate=False, has_side_effect=False),
}
show_table([{"절차": name, "자리": where_does_it_belong(**flags)} for name, flags in SCENARIOS.items()])

**관찰할 것.** 표의 결과가 맞는지보다 **어떤 기준이 예였는지**를 보세요. 첫 줄은 네 기준이 모두 예입니다. 어느 하나만 예여도 서버로 가는 규칙이라 경계 사례(예: 상태는 없지만 부작용만 있는 절차)에서 서버 쪽으로 기웁니다. 이 규칙이 틀리는 곳은 "서버를 띄우는 비용"이 큰 경우입니다. 절차 하나 때문에 서버를 새로 만드는 것은 과합니다. 기준에 비용 항목이 없다는 것이 이 휴리스틱의 약점입니다.

### 06 미리보기: Trigger Eval과 Quality Eval은 다른 질문이다

방금 본 12회의 점수 가운데 `trigger_ok`는 **Skill이 읽혔는가**를 묻고, `tool_selection_ok`는 **필요한 도구를 불렀는가**를 묻습니다. 둘 다 "일이 제대로 됐는가"(Quality Eval)가 아닙니다. 아래 칸은 두 검사가 실제로 무엇을 보는지 코드를 직접 열고, 12회를 두 축으로 나눠 셉니다.

In [ ]:
# [따라하기] trigger_ok와 tool_selection_ok가 보는 것을 코드로 읽고, 12회를 두 축으로 센다 (모델 호출 0회)
from cap_lab.scorers import tool_selection_ok, trigger_ok

print(inspect.getsource(trigger_ok))
print(inspect.getsource(tool_selection_ok))

if compare_rows:
    cross = {}
    for row in compare_rows:
        key = (row["variant"], f"trigger_ok={row['trigger_ok']}", f"tool_selection_ok={row['tool_selection_ok']}")
        cross[key] = cross.get(key, 0) + 1
    show_table([{"variant": v, "Trigger": t, "Tool 선택": s, "실행 수": n} for (v, t, s), n in sorted(cross.items())])
else:
    print("compare_rows가 비어 있어 교차표를 만들 수 없습니다.")

**관찰할 것.** `trigger_ok`는 에이전트가 `read_file`로 읽은 경로 목록(`record.skill_reads`)에 SKILL.md가 있는지만 봅니다. 읽은 **내용을 이해했는지**는 보지 않습니다. `tool_selection_ok`도 도구 이름만 봅니다. 인자가 맞는지(`args_valid`), 목표가 유지됐는지(`goal_preserved`), 결과가 확정되고 근거가 연결됐는지(`frozen_ok`, `evidence_linked`)는 **다른 검사**입니다. 교차표에서 `trigger_ok=False`인데 `tool_selection_ok=True`인 행이 있다면, 그것이 바로 둘이 다른 질문이라는 증거입니다.

- **Trigger Eval**: 올바른 때에 Skill이 읽혔는가. 입력은 사용자 요청, 판정 기준은 읽은 경로. 고장 나면 "아예 시작을 못 함" 또는 "필요 없는데 시작함".
- **Quality Eval**: Skill을 읽은 뒤 일이 올바르게 끝났는가. 인자, 확정, 근거 연결, 오류 대응. 고장 나면 "시작은 했는데 결과가 틀림".

둘을 섞으면 진단이 어려워집니다. 3절의 `include_tools` 고장과 이번 절의 교차표를 같이 보세요. Trigger는 통과하고 Tool 선택에서 실패했다면 원인은 도구 공개 쪽을 먼저 의심하게 됩니다. 06에서 이 분리를 각 단계의 결정적 scorer로 만들어 봅니다.

## 직접 해보기

세 가지를 제안합니다. 해보기 1은 모델을 부르지 않습니다. 해보기 2와 3은 유료이므로 기본으로 꺼져 있고, 켜는 것은 수강생 자신의 결정입니다. 켜기 전에 각 칸이 알려 주는 규모를 읽으세요.

### 해보기 1. 내 Skill로 공개 범위 바꾸기 (모델 호출 0회)

`include_tools`에 도구를 일부만 적은 Skill 사본을 만들고, 그 Skill을 읽었을 때 어떤 도구가 공개되는지 확인합니다. 이름 하나를 일부러 틀리게 적으면 어떻게 되는지도 봅니다. 아래 `MY_INCLUDE_TOOLS`만 고치면 됩니다. 이 변경은 작업 폴더의 사본에만 쓰이며 `skills/`는 그대로입니다.

In [ ]:
# [내 것 만들기] include_tools를 내가 고른 도구로 바꾸고 공개되는 도구를 확인한다 (모델 호출 0회)
MY_INCLUDE_TOOLS = "start_interview get_session list_evidence"  # 공백으로 구분한 문자열. 이름 하나를 틀리게 써 보세요.

# 틀린 이름이 있으면 어떤 이름인지 먼저 알려 준다. 서버 도구 이름은 1절에서 받아 둔 mcp_names다.
unknown_names = [name for name in MY_INCLUDE_TOOLS.split() if name not in mcp_names]
print("서버에 없는 이름:", unknown_names or "없음")

my_root = write_skill_copy("my-skill", description=original_description,
                           include_tools_lines=f'  include_tools: "{MY_INCLUDE_TOOLS}"')
with capture_skill_warnings() as warnings:
    my_middleware, my_state = load_skills(my_root)
my_text = (my_root / "requirements-interview" / "SKILL.md").read_text(encoding="utf-8")
request = await request_seen_by_model(my_middleware, my_state, conversation_after_reading(SKILL_PATH, my_text),
                                      deferred_tools)
report("내 Skill, 읽은 뒤", request)
print("경고:", warnings or "없음")

읽은 뒤 공개된 도구가 `MY_INCLUDE_TOOLS`에 적은 것과 같은지 확인하세요. 적지 않은 도구는 `tools`에 `defer_loading=True`로 남아 있지만 대화에는 정의가 끼워지지 않았습니다. 이때 모델이 그 도구 이름을 알고 부르면 어떻게 될까요? 4절의 결론을 이 상황에 적용해 한 문장으로 적어 보세요.

### 해보기 2. 다른 사례로 같은 비교 (유료, 약 110회)

5절의 비교를 dev-03과 dev-04(목표 문장을 그대로 `goal`에 넣어야 하는 사례)로 반복합니다. 이 사례들은 Skill을 읽었는지에 기대값이 없어서 `trigger_ok`가 `None`(확인 불가)입니다. 대신 `tool_selection_ok`를 봅니다. 아래 `RUN_EXERCISE`를 `True`로 바꿔야 실행됩니다.

In [ ]:
# [바꿔보기] 다른 사례(dev-03, dev-04)로 skill 공개 대 upfront 공개를 비교한다
RUN_EXERCISE = False  # True로 바꾸면 Claude를 약 110회 부릅니다
EXERCISE_CASES = ["dev-03", "dev-04"]

if RUN_EXERCISE:
    announce("해보기 2: 다른 사례로 공개 방식 비교", model_calls=110, runs=12)
    exercise_rows = run_small_compare(
        [Variant(name="skill-ex", tool_disclosure="skill"), Variant(name="upfront-ex", tool_disclosure="upfront")],
        EXERCISE_CASES, 3, label="part5-exercise",
    )
    show_table(pass_rate_table(exercise_rows, ["tool_selection_ok"]))
else:
    print("RUN_EXERCISE = False 라서 건너뜁니다. 호출 0회.")

**관찰할 것.** (켰다면) 5절의 표와 같은 순서로 읽으세요. 오류 행과 `unchecked`, 그다음 `passed/checked`와 구간, 마지막에 행 단위. `trigger_ok`가 `None`이라 구간이 비어 있는 것은 오류가 아니라 **검사 대상이 아니라는 뜻**입니다.

### 해보기 3. 더 많이 반복해서 구간 좁히기 (유료, 선택, 약 180회)

5절과 같은 비교를 5회씩 돌립니다(2 사례 x 5회 x 2 구성 = 20회). 구간이 얼마나 좁아지는지, 그래도 겹치는지 확인합니다. 결과를 본 뒤 반복을 더 늘려 원하는 결론을 만드는 것은 하지 않습니다. 몇 회로 할지는 실행 전에 정합니다.

In [ ]:
# [바꿔보기] 5회씩 반복해 구간이 얼마나 좁아지는지 본다
RUN_OPTIONAL = False  # True로 바꾸면 Claude를 약 180회 부릅니다 (20회 실행)
OPTIONAL_REPEATS = 5

if RUN_OPTIONAL:
    announce("해보기 3: 5회 반복 비교", model_calls=180, runs=20)
    optional_rows = run_small_compare(
        [Variant(name="skill-5", tool_disclosure="skill"), Variant(name="upfront-5", tool_disclosure="upfront")],
        ["dev-01", "dev-02"], OPTIONAL_REPEATS, label="part5-repeat5",
    )
    show_table(pass_rate_table(optional_rows, ["trigger_ok", "tool_selection_ok"]))
else:
    print("RUN_OPTIONAL = False 라서 건너뜁니다. 호출 0회.")

**관찰할 것.** (켰다면) 6/6에서 10/10으로 늘면 Wilson 하한이 약 0.61에서 약 0.72로 올라갑니다. 표본이 두 배 가까이 늘어도 구간은 이만큼만 좁아집니다. 구성 간 차이가 이 구간의 폭보다 작으면, 반복을 더 늘리는 것이 아니라 사전에 비교 방법(A/A 잡음 폭)을 정하는 것이 답입니다. 07의 주제입니다.

## 확인 질문

1. `metadata.include_tools`를 YAML 목록으로 쓰면 왜 도구가 하나도 공개되지 않습니까? 그때 deepagents는 예외를 내나요, 경고를 남기나요? 경고는 어디로 가나요?
2. `defer_loading`이 켜진 MCP 도구를 모델이 SKILL.md를 읽기 전에 이름으로 부르면 호출이 막히나요? 막힌다고 또는 안 막힌다고 답한다면 그 근거는 이 노트북의 어느 칸이고, 직접 실행해서 확인한 부분과 코드와 시험에 의존한 부분은 각각 무엇인가요?
3. description이 1024자를 넘을 때 잘려 나가는 것은 앞입니까, 뒤입니까? 이 Skill의 description에서 뒤가 잘리면 어떤 사례(dev-01, dev-02)가 먼저 위험해지고 그 이유는 무엇입니까?
4. 5절의 12회 결과에서 `skill`과 `upfront`의 통과율이 달라 보였다면, 왜 "skill 공개가 낫다"고 결론 내릴 수 없습니까? Wilson 구간, 오류 행, 사전 등록 규칙 중 두 가지로 설명하세요.
5. 어떤 절차를 MCP 서버에 두고 어떤 절차를 Skill 폴더의 스크립트에 둘지 기준 두 가지를 말하고, 이 과정의 에이전트가 Skill 스크립트를 실행할 수 없는 이유를 `cap_lab/agent.py:46`로 설명하세요.
6. Trigger Eval과 Quality Eval의 입력과 판정 기준은 각각 무엇입니까? `trigger_ok=False`인데 `tool_selection_ok=True`인 실행은 어떤 상황이며, 그 실행을 실패로 볼지 말지는 어느 구성(`skill`/`upfront`)이냐에 따라 왜 다릅니까?

## 정리

- Skill의 `description`은 모델이 Skill을 읽을지 정하는 유일한 근거이고 1024자까지만 남습니다. 쓰지 않는 조건을 앞에 둡니다.
- `metadata.include_tools`는 **공백으로 구분한 문자열**이어야 하며, 틀려도 예외 없이 경고만 남고 도구가 공개되지 않습니다. 이 과정은 `SkillToolResolver`를 쓰지 않습니다.
- `ChatAnthropic`에서는 Skill을 읽으면 요청의 `tools` 목록이 아니라 대화에 `tool_addition` 블록이 끼워집니다. 읽었다는 인정은 느슨합니다(부분 읽기도 인정, 오류는 불인정).
- **공개는 권한이 아닙니다.** 지연 공개는 스키마를 감출 뿐이고, 권한은 서버나 미들웨어, 사람 승인 같은 실행 지점에서 강제합니다. Skill에 적은 지침은 부탁입니다.
- skill 공개 대 upfront 공개를 12회 돌려 봤습니다. 구간이 겹치는 작은 표본이므로 결론이 아니라 관찰입니다. 오류 행, `unchecked`, 행 단위 기록을 함께 읽습니다.
- 상태·검증·부작용이 있는 절차는 MCP 서버에, 순수한 변환은 Skill 참고 자료에 둡니다. 이 과정의 에이전트는 파일 읽기 도구만 있어 스크립트를 실행하지 못합니다.
- Trigger Eval(읽혔는가)과 Quality Eval(일이 맞는가)은 다른 질문입니다. 06에서 각각의 scorer를 나눠 봅니다.

이 노트북에서 직접 확인하지 못한 것: 모델이 SKILL.md를 읽기 전에 deferred 도구를 부르는 에이전트 실행(시험과 코드로만 근거), `SkillsMiddleware(tools=...)`의 호출 차단 동작, 서버를 Streamable HTTP로 띄웠을 때의 공개 방식.